In [ ]:
# FSDL Lab 7 | Web Deployment with Gradio


!pip install gradio pytorch-lightning --quiet

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Gradio version: {gr.__version__}")
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 75.8 MB/s eta 0:00:00
Gradio version: 5.50.0
Device: cuda


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
    def forward(self, x):
        return self.block(x)

class EMNISTClassifier(nn.Module):
    def __init__(self, num_classes=26):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# train a quick model to use in the app
import torchvision
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.EMNIST(
    root='./data', split='letters',
    train=True, download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)

model = EMNISTClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Training model for 3 epochs...")
model.train()
for epoch in range(3):
    correct, total = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), (y-1).to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    print(f"Epoch {epoch+1}/3 | Acc: {correct/total:.4f}")

torch.save(model.state_dict(), "emnist_model.pth")
print("Model saved.")

100%|██████████| 562M/562M [00:03<00:00, 143MB/s]


Training model for 3 epochs...
Epoch 1/3 | Acc: 0.8595
Epoch 2/3 | Acc: 0.9187
Epoch 3/3 | Acc: 0.9318
Model saved.


In [ ]:
# reload model clean
inference_model = EMNISTClassifier().to(device)
inference_model.load_state_dict(torch.load("emnist_model.pth", map_location=device))
inference_model.eval()

transform_infer = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

def predict_letter(image):
    if image is None:
        return "No image provided"

    img = Image.fromarray(image.astype(np.uint8))
    img_tensor = transform_infer(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = inference_model(img_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        top3_probs, top3_idx = torch.topk(probs, 3)

    result = {}
    for prob, idx in zip(top3_probs, top3_idx):
        letter = chr(65 + idx.item())
        result[letter] = float(prob)

    return result

# test prediction
import torchvision
test_dataset = torchvision.datasets.EMNIST(
    root='./data', split='letters',
    train=False, download=True,
    transform=transforms.ToTensor()
)
test_img, test_label = test_dataset[0]
test_img_np = (test_img.squeeze().numpy() * 255).astype(np.uint8)
print(f"True label: {chr(64 + test_label)}")
print(f"Prediction: {predict_letter(test_img_np)}")

True label: A
Prediction: {'A': 0.8905779123306274, 'G': 0.04025611653923988, 'Q': 0.029072396457195282}


In [ ]:
with gr.Blocks(title="EMNIST Letter Classifier") as app:
    gr.Markdown("# ✍️ Handwritten Letter Classifier")
    gr.Markdown("Draw or upload a handwritten letter (A-Z) and the model will predict it.")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                label="Draw or upload a letter",
                type="numpy",
                height=280
            )
            predict_btn = gr.Button("Predict", variant="primary")

        with gr.Column():
            label_output = gr.Label(
                label="Top 3 Predictions",
                num_top_classes=3
            )

    predict_btn.click(
        fn=predict_letter,
        inputs=image_input,
        outputs=label_output
    )

    gr.Markdown("### How it works")
    gr.Markdown(
        "Model: CNN trained on EMNIST Letters (88,800 handwritten A-Z images). "
        "Draw a letter in the box above or upload an image and click Predict."
    )

app.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74b5c63f634ccf44f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


 FSDL Lab 7 Recap — Web Deployment with Gradio

## What We Built
A live web app wrapping our EMNIST letter classifier —
user draws or uploads a handwritten letter, model predicts top 3 letters.
Same concept as FSDL's TorchScript + Gradio deployment of their text recognizer.

## The Process
1. Trained EMNIST CNN for 3 epochs and saved weights as .pth file
2. Built a preprocessing pipeline matching training transforms
3. Wrote a predict_letter() function — takes image array, returns top 3 predictions
4. Wrapped everything in a Gradio Blocks interface
5. Launched with share=True — generates a public URL anyone can access

## Key Concepts Practiced
- Saving and loading PyTorch model weights
- Building an inference pipeline separate from training
- Gradio Blocks for clean UI layout
- share=True for instant public deployment — no server needed
- Softmax probabilities → top-k predictions for user-friendly output

## Why Gradio
Zero infrastructure needed — one line launches a web server with
a public URL. Perfect for demos, sharing with non-technical stakeholders,
and rapid prototyping before moving to production deployment.